# Riot API Data Collector

In [1]:
import requests, time, json, datetime
import pandas as pd
from tqdm.notebook import tqdm
from config import *
import random

In [4]:
# ── API 工具函数 ───────────────────────────────────
def api_get(url, params=None, retries=6):
    for attempt in range(retries):
        res = requests.get(url, headers=HEADERS, params=params or {})
        if res.status_code == 200:
            return res.json()
        elif res.status_code == 429:
            wait = int(res.headers.get("Retry-After", 2 ** (attempt + 1)))
            print(f"  [429] waiting {wait}s …")
            time.sleep(wait)
        elif res.status_code == 404:
            return None
        else:
            time.sleep(1.5)
    return None

def fetch_puuids_for_tier(tier: str, n_per_division: int) -> list:
    print(f"  Fetching {tier} …")
    puuids, seen = [], set()

    for div in ["I", "II", "III", "IV"]:
        collected = 0
        page = 1

        while collected < n_per_division:
            url = f"https://{REGION}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{div}"
            entries = api_get(url, params={"page": page})
            time.sleep(0.6)

            if not entries:  # 没数据了就停
                break

            random.shuffle(entries)

            for entry in entries:
                if collected >= n_per_division:
                    break
                puuid = entry.get("puuid")
                if puuid and puuid not in seen:
                    puuids.append(puuid)
                    seen.add(puuid)
                    collected += 1

            page += 1

        print(f"    {tier} {div}: {collected} players")

    print(f"  ✓ {len(puuids)} PUUIDs collected for {tier}")
    return puuids

def get_match_ids(puuid, count=100):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    data = api_get(url, params={"start": 0, "count": count})
    time.sleep(0.6)
    return data if isinstance(data, list) else []

def get_match_data(match_id):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    data = api_get(url)
    time.sleep(0.6)
    if data is None:
        return None, None
    t = data["info"].get("gameEndTimestamp")
    return t, data

# ── 主循环 ─────────────────────────────────────────
all_stats  = []
raw_total  = []   # 完整 match JSON，所有比赛
raw_valid  = []   # 完整 match JSON，Oct 22 前

for tier in TIERS:
    print(f"\n{'='*50}\n  TIER: {tier}\n{'='*50}")
    puuids = fetch_puuids_for_tier(tier, n_per_division=SAMPLES_PER_TIER)

    for puuid in tqdm(puuids, desc=f"{tier} players"):
        match_ids = get_match_ids(puuid, MATCHES_PER_PLAYER)
        if not match_ids:
            continue

        player_total, player_valid = 0, 0
        valid_ids = []

        for mid in match_ids:
            t, data = get_match_data(mid)
            if t is None or data is None:
                continue

            player_total += 1
            raw_total.append(data)

            if t < CUTOFF:
                player_valid += 1
                raw_valid.append(data)
                valid_ids.append(mid)

        all_stats.append({
            "puuid":    puuid,
            "tier":     tier,
            "total":    player_total,
            "valid":    player_valid,
            "filtered": player_total - player_valid,
            "valid_ids": valid_ids,
        })

    # ✅ 每个 tier 跑完立刻存一次，防止崩了全没
    with open(f"raw_valid_{tier}.json", "w") as f:
        json.dump(raw_valid, f)
    print(f"  ✅ {tier} saved → raw_valid_{tier}.json")

    with open(f"raw_total_{tier}.json", "w") as f:
        json.dump(raw_total, f)
    print(f"  ✅ {tier} saved → raw_total_{tier}.json")

# ── 汇总 ───────────────────────────────────────────
total_all = sum(r["total"] for r in all_stats)
valid_all = sum(r["valid"] for r in all_stats)

print(f"\n====== FINAL RESULT ======")
print(f"Total matches:  {total_all}")
print(f"Before Oct 22:  {valid_all}")
print(f"Keep ratio:     {round(valid_all/total_all, 3) if total_all else 0}")

# ── 存全量 JSON ────────────────────────────────────
with open("raw_valid_all.json", "w") as f:
    json.dump(raw_valid, f)
with open("all_stats.json", "w") as f:
    json.dump(all_stats, f)

print("✅ Done. raw_valid_all.json / all_stats.json saved.")


  TIER: IRON
  Fetching IRON …
    IRON I: 10 players
    IRON II: 10 players
    IRON III: 10 players
    IRON IV: 10 players
  ✓ 40 PUUIDs collected for IRON


IRON players:   0%|          | 0/40 [00:00<?, ?it/s]

  [429] waiting 32s …
  [429] waiting 32s …
  [429] waiting 31s …
  [429] waiting 23s …
  ✅ IRON saved → raw_valid_IRON.json
  ✅ IRON saved → raw_total_IRON.json

  TIER: BRONZE
  Fetching BRONZE …
    BRONZE I: 10 players
    BRONZE II: 10 players
    BRONZE III: 10 players
    BRONZE IV: 10 players
  ✓ 40 PUUIDs collected for BRONZE


BRONZE players:   0%|          | 0/40 [00:00<?, ?it/s]

  [429] waiting 22s …
  [429] waiting 31s …
  [429] waiting 30s …
  [429] waiting 32s …
  ✅ BRONZE saved → raw_valid_BRONZE.json
  ✅ BRONZE saved → raw_total_BRONZE.json

  TIER: SILVER
  Fetching SILVER …
    SILVER I: 10 players
    SILVER II: 10 players
    SILVER III: 10 players
    SILVER IV: 10 players
  ✓ 40 PUUIDs collected for SILVER


...

  [429] waiting 17s …
  [429] waiting 34s …
  [429] waiting 34s …
  [429] waiting 32s …
  ✅ GOLD saved → raw_valid_GOLD.json
  ✅ GOLD saved → raw_total_GOLD.json

  TIER: PLATINUM
  Fetching PLATINUM …
    PLATINUM I: 10 players
    PLATINUM II: 10 players
    PLATINUM III: 10 players
    PLATINUM IV: 10 players
  ✓ 40 PUUIDs collected for PLATINUM


PLATINUM players:   0%|          | 0/40 [00:00<?, ?it/s]

  [429] waiting 16s …
  [429] waiting 33s …
  [429] waiting 33s …
  [429] waiting 33s …
  [429] waiting 32s …
  ✅ PLATINUM saved → raw_valid_PLATINUM.json
  ✅ PLATINUM saved → raw_total_PLATINUM.json

  TIER: DIAMOND
  Fetching DIAMOND …
    DIAMOND I: 10 players
    DIAMOND II: 10 players
    DIAMOND III: 10 players
    DIAMOND IV: 10 players
  ✓ 40 PUUIDs collected for DIAMOND


DIAMOND players:   0%|          | 0/40 [00:00<?, ?it/s]

  [429] waiting 13s …
  [429] waiting 33s …
  [429] waiting 32s …
  [429] waiting 32s …
  ✅ DIAMOND saved → raw_valid_DIAMOND.json
  ✅ DIAMOND saved → raw_total_DIAMOND.json

====== FINAL RESULT ======
Total matches:  2397
Before Oct 22:  10
Keep ratio:     0.004
✅ Done. raw_valid_all.json / all_stats.json saved.
